# Phase 3: TrafficGNN — Graph Neural Network with Event Text

Architecture:
- Per-node temporal encoding (Conv1D over 15-step history)
- Text embedding via MiniLM → projected to model dimension
- GCN layers over road adjacency with text fusion
- MLP decoder → 3 horizons

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency,
    build_windows, compute_norm_stats, normalize, denormalize,
    train_val_split, compute_mse, write_submission,
)
from src.model import TrafficGNN
from src.train import (
    build_adj_tensor, encode_texts, create_dataloaders, train_model, evaluate,
)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Load & Prepare Data

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj = load_adjacency()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = np.array(T1 + T2)

mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
X_all_norm = normalize(X_all, mean, std)
Y_all_norm = normalize(Y_all, mean, std)

print(f"X: {X_all_norm.shape}, Y: {Y_all_norm.shape}")
print(f"X range: [{X_all_norm.min():.2f}, {X_all_norm.max():.2f}]")

In [ ]:
X_tr, _, Y_tr, X_va, _, Y_va = train_val_split(X_all_norm, None, Y_all_norm, val_frac=0.2)
T_tr, T_va = T_all[:len(X_tr)], T_all[len(X_tr):]
print(f"Train: {X_tr.shape}, Val: {X_va.shape}")

## 2. Encode Text (MiniLM)

In [ ]:
T_emb_tr = encode_texts(T_tr.tolist())
T_emb_va = encode_texts(T_va.tolist())
print(f"Train text emb: {T_emb_tr.shape}, Val: {T_emb_va.shape}")

## 3. Train Model

In [ ]:
adj_t = build_adj_tensor(adj).to(DEVICE)
train_loader, val_loader = create_dataloaders(
    X_tr, T_emb_tr, Y_tr, X_va, T_emb_va, Y_va, batch_size=32
)

model = TrafficGNN(
    num_nodes=1260, in_steps=15, out_steps=3,
    text_dim=384, d_model=64, num_layers=4, dropout=0.2,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

In [ ]:
best_val = train_model(
    model, train_loader, val_loader, adj_t, DEVICE,
    epochs=100, lr=1e-3, patience=15,
)
print(f"Best val loss (normalized): {best_val:.4f}")

## 4. Evaluate

In [ ]:
@torch.no_grad()
def predict(model, X, T_emb, adj_t, device, batch_size=64):
    model.eval()
    preds = []
    for i in range(0, len(X), batch_size):
        xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32, device=device)
        tb = torch.tensor(T_emb[i:i+batch_size], dtype=torch.float32, device=device)
        pb = model(xb, tb, adj_t)
        preds.append(pb.cpu().numpy())
    return np.concatenate(preds, axis=0)

y_pred_norm = predict(model, X_va, T_emb_va, adj_t, DEVICE)
y_pred = denormalize(y_pred_norm, mean, std)
y_true = denormalize(Y_va, mean, std)

mse_gnn = compute_mse(y_pred, y_true)
print(f"TrafficGNN Val MSE: {mse_gnn:.4f}")

for hi, hn in enumerate(["h5", "h10", "h15"]):
    mse_h = compute_mse(y_pred[:, hi:hi+1, :], y_true[:, hi:hi+1, :])
    print(f"  {hn}: {mse_h:.4f}")

## 5. Retrain on All Data & Submit

In [ ]:
T_emb_all = encode_texts(T_all.tolist())

full_ds = torch.utils.data.TensorDataset(
    torch.tensor(X_all_norm),
    torch.tensor(T_emb_all),
    torch.tensor(Y_all_norm),
)
full_loader = torch.utils.data.DataLoader(full_ds, batch_size=32, shuffle=True)

model_full = TrafficGNN(
    num_nodes=1260, in_steps=15, out_steps=3,
    text_dim=384, d_model=64, num_layers=4, dropout=0.2,
).to(DEVICE)

opt = torch.optim.Adam(model_full.parameters(), lr=1e-3)
crit = torch.nn.MSELoss()
for epoch in range(1, 51):
    model_full.train()
    total = 0
    for Xb, Tb, Yb in full_loader:
        Xb, Tb, Yb = Xb.to(DEVICE), Tb.to(DEVICE), Yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(model_full(Xb, Tb, adj_t), Yb)
        loss.backward()
        opt.step()
        total += loss.item() * Xb.size(0)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss: {total / len(full_ds):.4f}")

In [ ]:
test_hist, test_texts = load_test_data()
test_hist_norm = normalize(test_hist, mean, std)
test_emb = encode_texts(test_texts)

test_preds_norm = predict(model_full, test_hist_norm, test_emb, adj_t, DEVICE)
test_preds = denormalize(test_preds_norm, mean, std)
test_preds = test_preds.clip(min=0)  # speed cannot be negative

write_submission(test_preds, label="gnn_text", models={"gnn": model_full.state_dict()})